# Phase 5 — Fraud Detection Analysis

This notebook explores the Banking AI Copilot fraud detection pipeline:
1. Dataset exploration and class distribution
2. Feature engineering review
3. Model training (XGBoost + ensemble)
4. SHAP feature attribution
5. Model comparison across all 5 models
6. Real-time prediction demo

**Run from the project root:** `jupyter notebook notebooks/02_fraud_detection_analysis.ipynb`

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

os.environ.setdefault('DATABASE_URL', 'sqlite:///../banking.db')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.figsize'] = (12, 6)
matplotlib.rcParams['axes.grid'] = True

print('Libraries loaded.')

## 1. Dataset Exploration

In [ ]:
df = pd.read_csv('../data/processed/enriched_transactions.csv')
print(f'Total transactions: {len(df):,}')
print(f'Fraud rate: {df["is_fraud"].mean():.2%}')
print(f'\nClass distribution:')
print(df['is_fraud'].value_counts())
print(f'\nFeature columns: {list(df.columns)}')

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
features = ['amount', 'hour', 'amount_zscore', 'velocity_30m', 'credit_score', 'merchant_risk']

for i, feat in enumerate(features):
    ax = axes[i // 3][i % 3]
    df[df['is_fraud'] == 0][feat].hist(ax=ax, alpha=0.6, label='Legitimate', bins=30, color='steelblue')
    df[df['is_fraud'] == 1][feat].hist(ax=ax, alpha=0.6, label='Fraud', bins=30, color='tomato')
    ax.set_title(feat)
    ax.legend()

plt.suptitle('Feature Distributions: Fraud vs Legitimate', fontsize=14)
plt.tight_layout()
plt.show()

## 2. Model Training

In [ ]:
from src.models.fraud_detector import FEATURE_COLS, prepare_features, apply_smote
from sklearn.model_selection import train_test_split

X, y = prepare_features(df)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
X_train_smote, y_train_smote = apply_smote(X_train, y_train)

print(f'Train (after SMOTE): {len(X_train_smote):,}  |  Test: {len(X_test):,}')
print(f'SMOTE fraud rate: {y_train_smote.mean():.2%}')

## 3. Load Pre-trained Model & Evaluate

In [ ]:
from src.models.fraud_detector import load_model
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, auc

model = load_model('../models/fraud_detector_v1.pkl')
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print(classification_report(y_test, y_pred, target_names=['Legitimate', 'Fraud']))

precision, recall, _ = precision_recall_curve(y_test, y_prob)
pr_auc = auc(recall, precision)
roc_auc = roc_auc_score(y_test, y_prob)
print(f'PR-AUC:  {pr_auc:.4f}')
print(f'ROC-AUC: {roc_auc:.4f}')

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Precision-Recall curve
ax1.plot(recall, precision, color='blue', lw=2, label=f'XGBoost (PR-AUC={pr_auc:.3f})')
ax1.set_xlabel('Recall'); ax1.set_ylabel('Precision')
ax1.set_title('Precision-Recall Curve'); ax1.legend()

# Score distribution
ax2.hist(y_prob[y_test == 0], bins=40, alpha=0.6, label='Legitimate', color='steelblue')
ax2.hist(y_prob[y_test == 1], bins=40, alpha=0.6, label='Fraud', color='tomato')
ax2.axvline(0.5, color='black', linestyle='--', label='Threshold 0.5')
ax2.set_xlabel('Fraud Probability'); ax2.set_ylabel('Count')
ax2.set_title('Score Distribution'); ax2.legend()

plt.suptitle('XGBoost Fraud Detector Performance', fontsize=14)
plt.tight_layout()
plt.show()

## 4. SHAP Feature Attribution

In [ ]:
try:
    import shap
    explainer = shap.TreeExplainer(model)
    sample = X_test.sample(min(200, len(X_test)), random_state=42)
    shap_values = explainer.shap_values(sample)

    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, sample, feature_names=FEATURE_COLS, show=False)
    plt.title('SHAP Feature Importance')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('shap not installed — run: pip install shap')

## 5. Feature Importance (XGBoost Built-in)

In [ ]:
importances = pd.Series(model.feature_importances_, index=FEATURE_COLS)
importances.sort_values().plot(kind='barh', color='steelblue')
plt.title('XGBoost Feature Importance')
plt.xlabel('Importance Score')
plt.tight_layout()
plt.show()
print(importances.sort_values(ascending=False))

## 6. Real-time Prediction Demo

In [ ]:
from src.models.fraud_detector import fraud_predict

test_cases = [
    {'name': 'Suspicious — new device, late night, high amount',
     'txn': {'amount': 9500, 'hour': 2, 'geo_mismatch': 1, 'device_new': 1,
             'is_weekend': 0, 'amount_zscore': 4.5, 'velocity_30m': 6,
             'credit_score': 410, 'account_age_days': 12, 'merchant_risk': 0.95}},
    {'name': 'Normal — regular daytime purchase',
     'txn': {'amount': 85, 'hour': 14, 'geo_mismatch': 0, 'device_new': 0,
             'is_weekend': 0, 'amount_zscore': 0.2, 'velocity_30m': 1,
             'credit_score': 780, 'account_age_days': 2000, 'merchant_risk': 0.05}},
    {'name': 'Borderline — slightly unusual amount, weekend',
     'txn': {'amount': 1200, 'hour': 22, 'geo_mismatch': 0, 'device_new': 0,
             'is_weekend': 1, 'amount_zscore': 1.8, 'velocity_30m': 2,
             'credit_score': 650, 'account_age_days': 400, 'merchant_risk': 0.3}},
]

print(f'{"Case":<50} {"Prob":>8} {"Tier":<10}')
print('-' * 70)
for case in test_cases:
    result = fraud_predict(case['txn'], model)
    prob = result['fraud_probability']
    tier = result['risk_level']
    print(f'{case["name"]:<50} {prob:>8.2%} {tier:<10}')

## 7. Threshold Optimization

In [ ]:
from sklearn.metrics import f1_score

thresholds = np.arange(0.1, 0.9, 0.05)
f1s = [f1_score(y_test, (y_prob >= t).astype(int)) for t in thresholds]

best_t = thresholds[np.argmax(f1s)]
plt.plot(thresholds, f1s, marker='o', color='steelblue')
plt.axvline(best_t, color='red', linestyle='--', label=f'Best threshold: {best_t:.2f}')
plt.xlabel('Threshold'); plt.ylabel('F1 Score')
plt.title('F1 Score vs Classification Threshold')
plt.legend()
plt.tight_layout()
plt.show()
print(f'Best threshold: {best_t:.2f}  →  F1: {max(f1s):.4f}')